In [1]:
from openai import OpenAI
import os

or_client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY")
)

In [2]:
from datasets import load_dataset
ds = load_dataset("weathon/grpo_dataset")

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm 
import os
import json

os.makedirs("rollouts", exist_ok=True)

def rollout(idx):
    if os.path.exists(f"rollouts/{idx:05d}.json"):
        with open(f"rollouts/{idx:05d}.json", "r") as f:
            ans = json.load(f)
        return ans, 0
    
    # for each message, we roll out in sequence, but we parallelize across messages, to hit cache and make it cheaper
    messages = ds["train"][idx]["prompt"]
    messages[0]["content"] = messages[0]["content"]#.replace("Give your analysis first, then put your final score in a XML-style tag <score></score>", "Give a brief analysis first, then put your final score in a XML-style tag <score></score>. Do not think for too long.")
    cost = 0
    ans = []
    for _ in range(2):
        _response = or_client.chat.completions.create(
            model="deepseek/deepseek-v4-flash",
            messages=messages,
            temperature=1.2,
            extra_body={"reasoning": {"enabled": False, "effort": "low"}, "provider": {"only": ["deepseek"]}}
        )
        response = _response.choices[0].message.content
        ans.append(response)
        cost += _response.usage.cost_details["upstream_inference_cost"]

    for _ in range(2):
        _response = or_client.chat.completions.create(
            model="qwen/qwen3.5-flash-02-23",
            messages=messages,
            temperature=1.2,
            extra_body={"reasoning": {"enabled": False, "effort": "low"}}
        )
        response = _response.choices[0].message.content
        ans.append(response)
        cost += _response.usage.cost_details["upstream_inference_cost"]
    with open(f"rollouts/{idx:05d}.json", "w") as f:
        json.dump(ans, f, indent=4)

    return ans, cost



with ThreadPoolExecutor(max_workers=50) as executor:
    rollouts = list(tqdm(executor.map(rollout, range(len(ds["train"]))), total=len(ds["train"])))


  0%|          | 1/1894 [00:52<27:41:50, 52.67s/it]

In [ ]:
rollouts[0][0][0]

ChatCompletionMessage(content='The paper under review presents the **Perceptual Group Tokenizer (PGT)**, a visual recognition backbone built entirely from iterative perceptual grouping operations instead of standard convolutions or self-attention. It achieves 80.3% top-1 accuracy on ImageNet-1K linear probing, matching ViT-B/8 baselines, while offering unique advantages like adaptive computation (changing the number of group tokens at inference) and strong interpretability through multi-head grouping.\n\n**Comparison with Anchor Papers:**\n\n1.  **"Multi-Task Perception in Unstructured Environments"** (Scores: 1, 1, 3, 3): That paper was rated very poorly due to minimal novelty (direct combination of existing methods), missing experiments, and lack of comparisons. In contrast, PGT presents a genuinely novel architecture (a full grouping-based vision backbone) with thorough ablations, strong baselines, and clear demonstrations of its unique properties. PGT is vastly stronger.\n\n2.  **"

In [ ]:
import re
for i, rollout in enumerate(rollouts):
    print(f"Rollout {i}:")
    print([re.search('<score>(.*?)</score>', r.content).group(1) for r in rollout[0]])
    print(f"GT: {ds['train'][i]['solution']}")
    print()

Rollout 0:
['8', '6.5', '6.0', '8', '8.0', '7.5']
GT: 6.6

Rollout 1:
['6.5', '6.0', '6.5', '6.5', '8.0', '6.5']
GT: 6.0

Rollout 2:
['3', '3.5', '2', '3.0', '2.0', '3.5']
GT: 5.0

Rollout 3:
['6.0', '5.0', '5.5', '6.0', '6.0', '6.5']
GT: 4.0

Rollout 4:
['5.5', '4', '5.0', '7.5', '7.0', '7.5']
GT: 3.25

Rollout 5:
['5', '4.5', '6.5', '6.0', '8.5', '7.0']
GT: 4.75

Rollout 6:
['5.5', '4', '3.5', '6.5', '8.0', '7.0']
GT: 4.75

Rollout 7:
['2.0', '2.5', '4.5', '3.0', '3.0', '3.5']
GT: 4.0

Rollout 8:
['4.0', '3', '1', '3.5', '3.5', '6.0']
GT: 5.75

Rollout 9:
['7.0', '6', '6.0', '7.5', '8.5', '8.0']
GT: 4.0

Rollout 10:
['6', '5', '6.0', '3.0', '7.0', '4.5']
GT: 6.0

Rollout 11:
['6', '7.0', '6.5', '8.0', '6.5', '8.5']
GT: 5.25

Rollout 12:
['6', '6.0', '5.5', '7.5', '8.5', '8.5']
GT: 4.5

Rollout 13:
['7.0', '7.0', '7', '7.0', '9.0', '8.5']
GT: 7.0

Rollout 14:
['4.0', '5.0', '5.0', '3.5', '4.5', '3.5']
GT: 4.25

Rollout 15:
['6.0', '5.0', '4', '7.5', '7.5', '7.0']
GT: 4.75

Rollout 16: